# Step 05 — Analysis: Embeddings, t-SNE, Linear Probe, ENSO Modulation
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

Evaluates whether the Siamese CNN learned a representation that captures
how ENSO modulates BSISO atmospheric structure.

**Inputs (from Google Drive):**
- `checkpoints/encoder_final.pth` — trained CNN encoder weights
- `data/processed/X_July.npy` — shape `(N, 3, 31, 51)`, normalized
- `data/processed/labels_aligned.csv` — BSISO phase + ENSO category

**Analysis steps:**
1. Extract embeddings — run all N samples → `(N, 64)` matrix
2. t-SNE — 2D projection colored by phase and ENSO
3. Linear probe — logistic regression on embeddings
4. ENSO displacement analysis — El Niño vs La Niña centroids per phase
5. Nearest-neighbor retrieval — qualitative check

**Outputs (saved to `results/`):**
- `embeddings.npy` — `(N, 64)` float32
- `tsne_by_phase.png`, `tsne_by_enso.png`
- `linear_probe_results.json`
- `enso_displacement.png`
- `analysis_report.txt`

---

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
RESULTS_DIR    = f'{PROJECT_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Files in checkpoints/:')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    mb = os.path.getsize(f'{CHECKPOINT_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.2f} MB)')

## Cell 2 — Define CNN Encoder

Exact same architecture as notebook 04 — required to load the saved weights.

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc          = nn.Linear(128, embedding_dim)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).view(x.size(0), -1)
        z = self.fc(x)
        z = F.normalize(z, p=2, dim=1)
        return z

print(f'CNNEncoder defined.  Parameters: {sum(p.numel() for p in CNNEncoder().parameters()):,}')

## Cell 3 — Load Model + Extract All Embeddings

Run all N samples through the encoder in batches → `(N, 64)` matrix.  
Saved to `results/embeddings.npy` for reuse without re-running the model.

In [ ]:
# Load data
X      = np.load(f'{PROCESSED_DIR}/X_July.npy')          # (N, 3, 31, 51)
labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned.csv', parse_dates=['date'])
print(f'X shape:  {X.shape}')
print(f'Labels:   {len(labels)} rows')

# Load trained encoder
encoder = CNNEncoder(embedding_dim=64).to(device)
state_dict = torch.load(f'{CHECKPOINT_DIR}/encoder_final.pth', map_location=device)
encoder.load_state_dict(state_dict)
encoder.eval()
print(f'\nModel loaded from: encoder_final.pth')

# Extract embeddings in batches of 128
BATCH = 128
embeddings = np.zeros((len(X), 64), dtype=np.float32)

with torch.no_grad():
    for start in range(0, len(X), BATCH):
        end   = min(start + BATCH, len(X))
        batch = torch.from_numpy(X[start:end]).float().to(device)
        embeddings[start:end] = encoder(batch).cpu().numpy()

print(f'\nEmbeddings shape: {embeddings.shape}  (N, 64)')
print(f'Norms (first 5): {np.linalg.norm(embeddings[:5], axis=1)}  (should all be ~1.0)')

# Save
emb_path = f'{RESULTS_DIR}/embeddings.npy'
np.save(emb_path, embeddings)
print(f'Saved: {emb_path}')

## Cell 4 — t-SNE Visualization

Two side-by-side 2D projections of the 64-dim embedding space:
- **Left:** colored by BSISO phase (1–8) — should show 8 clusters if phase is encoded
- **Right:** colored by ENSO category — should show separation *within* phase clusters

This is the key qualitative result of the project.

In [ ]:
print('Running t-SNE (perplexity=30, n_iter=1000)...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
            random_state=42, n_jobs=-1)
Z = tsne.fit_transform(embeddings)    # (N, 2)
print(f't-SNE done. Shape: {Z.shape}')

# ── Plot 1: colored by BSISO phase ──────────────────────────────────────────
phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
enso_palette = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker  = {'El Nino': '^',       'Neutral': 'o',        'La Nina': 's'}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: by BSISO phase
ax = axes[0]
for phase in range(1, 9):
    mask = labels['bsiso_phase'] == phase
    ax.scatter(Z[mask, 0], Z[mask, 1],
               c=[phase_colors[phase - 1]], s=12, alpha=0.6, label=f'Phase {phase}')
ax.set_title('t-SNE colored by BSISO Phase', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
ax.legend(fontsize=9, markerscale=2, ncol=2, loc='best')
ax.grid(alpha=0.2)

# Right: by ENSO category
ax = axes[1]
for cat in ['El Nino', 'Neutral', 'La Nina']:
    mask = labels['enso_category'] == cat
    ax.scatter(Z[mask, 0], Z[mask, 1],
               c=enso_palette[cat], marker=enso_marker[cat],
               s=12, alpha=0.5, label=cat)
ax.set_title('t-SNE colored by ENSO Category', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
ax.legend(fontsize=11, markerscale=2, loc='best')
ax.grid(alpha=0.2)

plt.suptitle('Siamese CNN Embedding Space (64-dim → t-SNE 2D)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tsne_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: results/tsne_overview.png')

## Cell 4b — t-SNE: ENSO Separation Within Each Phase

8-panel plot — one panel per BSISO phase.  
Within each panel, points are colored by ENSO category.  
**Key question:** Are El Niño and La Niña points separated within each phase cluster?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('t-SNE: ENSO Separation Within Each BSISO Phase\n'
             'Red=El Niño  Grey=Neutral  Blue=La Niña',
             fontsize=13, fontweight='bold')

for phase in range(1, 9):
    ax   = axes[(phase - 1) // 4, (phase - 1) % 4]
    mask_phase = labels['bsiso_phase'] == phase

    for cat in ['El Nino', 'Neutral', 'La Nina']:
        mask = mask_phase & (labels['enso_category'] == cat)
        n = mask.sum()
        if n == 0:
            continue
        ax.scatter(Z[mask, 0], Z[mask, 1],
                   c=enso_palette[cat], marker=enso_marker[cat],
                   s=20, alpha=0.7, label=f'{cat} (n={n})')

    ax.set_title(f'Phase {phase}  (n={mask_phase.sum()})', fontsize=11)
    ax.legend(fontsize=7, markerscale=1.5)
    ax.grid(alpha=0.2)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tsne_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/tsne_by_phase.png')
print()
print('What to look for:')
print('  - If El Nino/La Nina points form distinct sub-clusters within each phase panel')
print('    → model learned ENSO modulation of BSISO structure')
print('  - If they are interleaved/random → model only learned BSISO phase, not ENSO')

## Cell 5 — Linear Probe

Train a simple logistic regression classifier on top of the frozen embeddings.  
This measures how much task-relevant information the encoder captured.

**Tasks:**
1. **Predict BSISO phase** (8 classes) — random baseline: 12.5%
2. **Predict ENSO category** (3 classes) — random baseline: ~33%, majority baseline: ~58% (Neutral)

High BSISO accuracy + meaningful ENSO accuracy above majority baseline = success.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Load train/val split from notebook 04
train_idx = np.load(f'{PROCESSED_DIR}/train_indices.npy')
val_idx   = np.load(f'{PROCESSED_DIR}/val_indices.npy')

Z_train = embeddings[train_idx]
Z_val   = embeddings[val_idx]

probe_results = {}

print('=' * 55)
print('LINEAR PROBE RESULTS')
print('=' * 55)

for task, col, baseline_label in [
    ('BSISO Phase',    'bsiso_phase',    'Random = 12.5%'),
    ('ENSO Category',  'enso_category',  'Majority = ~58% (Neutral)'),
]:
    y_train = labels.loc[train_idx, col].values
    y_val   = labels.loc[val_idx,   col].values

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(Z_train, y_train)

    train_acc = accuracy_score(y_train, clf.predict(Z_train))
    val_acc   = accuracy_score(y_val,   clf.predict(Z_val))

    # 5-fold CV on full dataset for robust estimate
    cv_scores = cross_val_score(clf, embeddings,
                                labels[col].values,
                                cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                scoring='accuracy', n_jobs=-1)

    probe_results[task] = {
        'train_acc': float(train_acc),
        'val_acc':   float(val_acc),
        'cv_mean':   float(cv_scores.mean()),
        'cv_std':    float(cv_scores.std()),
    }

    print(f'\nTask: {task}  ({baseline_label})')
    print(f'  Train accuracy:     {train_acc*100:.1f}%')
    print(f'  Val   accuracy:     {val_acc*100:.1f}%')
    print(f'  5-fold CV:          {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%')
    print(f'  Val classification report:')
    print(classification_report(y_val, clf.predict(Z_val), zero_division=0))

# Save
with open(f'{RESULTS_DIR}/linear_probe_results.json', 'w') as f:
    json.dump(probe_results, f, indent=2)
print('Saved: results/linear_probe_results.json')

## Cell 6 — ENSO Displacement Analysis

**Core research question:** Does ENSO modulate BSISO structure in embedding space?

Method: within each BSISO phase, compute the **centroid** of El Niño days and La Niña days
in the 64-dim embedding space. The displacement vector (El Niño centroid − La Niña centroid)
tells us how ENSO shifts the representation.

**Findings to look for:**
- Are the displacement magnitudes consistently large across all 8 phases? → ENSO is encoded
- Are the displacement directions consistent across phases? → ENSO modulation is universal
- Compare to a **random baseline** (shuffle ENSO labels) to check statistical significance

In [ ]:
# ── 1. Compute El Niño / La Niña centroids per phase ────────────────────────
phases      = range(1, 9)
disp_mag    = []   # ||centroid_EN - centroid_LN|| per phase
cos_EN_LN   = []   # cosine similarity between EN and LN centroids

print(f'{"Phase":<8} {"n(EN)":>6} {"n(LN)":>6} {"||EN-LN||":>10} {"cos(EN,LN)":>12}')
print('-' * 46)

for phase in phases:
    mask_EN = (labels['bsiso_phase'] == phase) & (labels['enso_category'] == 'El Nino')
    mask_LN = (labels['bsiso_phase'] == phase) & (labels['enso_category'] == 'La Nina')

    if mask_EN.sum() < 3 or mask_LN.sum() < 3:
        disp_mag.append(np.nan); cos_EN_LN.append(np.nan)
        print(f'{phase:<8} {mask_EN.sum():>6} {mask_LN.sum():>6}  (too few samples)')
        continue

    c_EN = embeddings[mask_EN].mean(axis=0)   # (64,)
    c_LN = embeddings[mask_LN].mean(axis=0)   # (64,)

    mag = np.linalg.norm(c_EN - c_LN)
    cos = np.dot(c_EN, c_LN) / (np.linalg.norm(c_EN) * np.linalg.norm(c_LN) + 1e-8)

    disp_mag.append(mag)
    cos_EN_LN.append(cos)
    print(f'{phase:<8} {mask_EN.sum():>6} {mask_LN.sum():>6} {mag:>10.4f} {cos:>12.4f}')

# ── 2. Random baseline (shuffle ENSO labels 100 times) ──────────────────────
rng          = np.random.default_rng(42)
baseline_mag = []
for _ in range(100):
    shuffled = labels['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
    mags_trial = []
    for phase in phases:
        mask_ph = (labels['bsiso_phase'] == phase).values
        mask_EN = mask_ph & (shuffled == 'El Nino')
        mask_LN = mask_ph & (shuffled == 'La Nina')
        if mask_EN.sum() < 3 or mask_LN.sum() < 3:
            continue
        c_EN = embeddings[mask_EN].mean(axis=0)
        c_LN = embeddings[mask_LN].mean(axis=0)
        mags_trial.append(np.linalg.norm(c_EN - c_LN))
    if mags_trial:
        baseline_mag.append(np.mean(mags_trial))

baseline_mean = np.mean(baseline_mag)
baseline_std  = np.std(baseline_mag)
observed_mean = np.nanmean(disp_mag)
z_score       = (observed_mean - baseline_mean) / (baseline_std + 1e-8)

print(f'\nEN–LN displacement summary:')
print(f'  Observed mean: {observed_mean:.4f}')
print(f'  Null baseline: {baseline_mean:.4f} ± {baseline_std:.4f}')
print(f'  Z-score:       {z_score:.2f}  (> 2 = statistically significant)')

# ── 3. Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
valid_phases = [p for p, m in zip(phases, disp_mag) if not np.isnan(m)]
valid_mags   = [m for m in disp_mag if not np.isnan(m)]
ax.bar(valid_phases, valid_mags, color='steelblue', alpha=0.8, label='Observed')
ax.axhline(baseline_mean, color='red',    linestyle='--', linewidth=1.5, label=f'Null mean ({baseline_mean:.3f})')
ax.axhline(baseline_mean + 2*baseline_std, color='red', linestyle=':', linewidth=1, label='Null +2σ')
ax.set_xlabel('BSISO Phase', fontsize=12)
ax.set_ylabel('||Centroid(El Niño) − Centroid(La Niña)||', fontsize=11)
ax.set_title('EN−LN Embedding Displacement per BSISO Phase', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 9))
ax.legend(fontsize=10); ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(valid_phases, valid_mags, color='steelblue', alpha=0.8)
ax.axhline(observed_mean, color='steelblue', linestyle='-',  linewidth=2, label=f'Observed mean ({observed_mean:.3f})')
ax.axhline(baseline_mean, color='red',       linestyle='--', linewidth=2, label=f'Null mean ({baseline_mean:.3f})')
ax.set_title(f'Z-score = {z_score:.2f}  (significant if > 2)', fontsize=12, fontweight='bold')
ax.set_xlabel('BSISO Phase', fontsize=12)
ax.set_ylabel('Displacement magnitude', fontsize=11)
ax.set_xticks(range(1, 9))
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enso_displacement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/enso_displacement.png')

## Cell 7 — Nearest-Neighbor Retrieval

Qualitative check: pick a query sample and find its **k nearest neighbors**
in embedding space. Verify that they share the same BSISO phase and ideally the same ENSO.

Retrieval precision = fraction of top-k neighbors with same label as query.

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

K = 10

# Precision@K: fraction of k-NN that share the same label as query
def precision_at_k(embeddings, label_array, k=10, n_queries=200, seed=42):
    rng   = np.random.default_rng(seed)
    query_idx = rng.choice(len(embeddings), size=n_queries, replace=False)
    D = cosine_distances(embeddings)   # (N, N)
    prec = []
    for q in query_idx:
        # Sort by distance, exclude self (index 0)
        nn_idx = np.argsort(D[q])[1:k+1]
        same   = np.mean(label_array[nn_idx] == label_array[q])
        prec.append(same)
    return np.mean(prec), np.std(prec)

print(f'Precision@{K} (fraction of {K}-NN with same label):')
print(f'  Random baseline = 1/8 = 12.5% (phase), ~33% (ENSO)')
print()

for task, col in [('BSISO Phase', 'bsiso_phase'), ('ENSO Category', 'enso_category')]:
    label_arr = labels[col].values
    mean_prec, std_prec = precision_at_k(embeddings, label_arr, k=K)
    print(f'  {task:<18}: {mean_prec*100:.1f}% ± {std_prec*100:.1f}%')

# ── Qualitative: show 5 example retrievals ───────────────────────────────────
print(f'\nExample retrievals (query → top {K} neighbors):')
print(f'{"Query Date":<14} {"Phase":<7} {"ENSO":<10} | {"Neighbors: (date, phase, ENSO) ..."}')
print('-' * 90)

D = cosine_distances(embeddings)
rng = np.random.default_rng(0)
for q in rng.choice(len(embeddings), size=5, replace=False):
    nn_idx  = np.argsort(D[q])[1:K+1]
    q_row   = labels.iloc[q]
    nn_info = [(str(labels.iloc[n]['date'])[:10],
                labels.iloc[n]['bsiso_phase'],
                labels.iloc[n]['enso_category'][:2]) for n in nn_idx[:5]]
    nn_str  = '  '.join([f"{d}|P{p}|{e}" for d, p, e in nn_info])
    print(f'{str(q_row["date"])[:10]:<14} P{q_row["bsiso_phase"]:<6} {q_row["enso_category"]:<10} | {nn_str}')

## Cell 8 — Summary Report

In [ ]:
# Reload probe results in case cell 5 was run separately
with open(f'{RESULTS_DIR}/linear_probe_results.json') as f:
    probe_results = json.load(f)

report_lines = [
    '=' * 60,
    'ANALYSIS SUMMARY REPORT',
    '=' * 60,
    f'Model:       encoder_final.pth',
    f'Embedding:   (N={len(embeddings)}, dim=64), L2-normalized',
    '',
    '── Linear Probe ────────────────────────────────────────',
]
for task, res in probe_results.items():
    report_lines += [
        f'{task}:',
        f'  Val accuracy:  {res["val_acc"]*100:.1f}%',
        f'  5-fold CV:     {res["cv_mean"]*100:.1f}% ± {res["cv_std"]*100:.1f}%',
    ]

report_lines += [
    '',
    '── ENSO Displacement ───────────────────────────────────',
    f'Observed mean ||EN-LN|| per phase: {observed_mean:.4f}',
    f'Null baseline:                     {baseline_mean:.4f} ± {baseline_std:.4f}',
    f'Z-score:                           {z_score:.2f}',
    '',
    '── Interpretation ──────────────────────────────────────',
]

phase_acc = probe_results.get('BSISO Phase', {}).get('val_acc', 0)
enso_acc  = probe_results.get('ENSO Category', {}).get('val_acc', 0)

if phase_acc > 0.5:
    report_lines.append(f'BSISO phase: STRONGLY encoded  ({phase_acc*100:.0f}% >> 12.5% baseline)')
elif phase_acc > 0.25:
    report_lines.append(f'BSISO phase: WEAKLY encoded    ({phase_acc*100:.0f}% > 12.5% baseline)')
else:
    report_lines.append(f'BSISO phase: NOT encoded       ({phase_acc*100:.0f}% ~ baseline)')

if enso_acc > 0.65:
    report_lines.append(f'ENSO:        STRONGLY encoded  ({enso_acc*100:.0f}% >> 58% majority baseline)')
elif enso_acc > 0.58:
    report_lines.append(f'ENSO:        WEAKLY encoded    ({enso_acc*100:.0f}% > 58% majority baseline)')
else:
    report_lines.append(f'ENSO:        NOT encoded       ({enso_acc*100:.0f}% ~ majority baseline)')

if z_score > 2:
    report_lines.append(f'ENSO displacement: SIGNIFICANT (z={z_score:.2f}) — model separated EN/LN in embedding space')
else:
    report_lines.append(f'ENSO displacement: NOT significant (z={z_score:.2f}) — EN/LN not separated')

report_lines.append('=' * 60)

report = '\n'.join(report_lines)
print(report)

with open(f'{RESULTS_DIR}/analysis_report.txt', 'w') as f:
    f.write(report)
print(f'\nSaved: results/analysis_report.txt')

---
## Done!

Results saved to Google Drive:
```
BSISO_SSL_Project/results/
├── embeddings.npy            ← (N, 64) all embeddings
├── tsne_overview.png         ← t-SNE colored by phase (left) and ENSO (right)
├── tsne_by_phase.png         ← 8-panel t-SNE, ENSO color within each phase
├── enso_displacement.png     ← EN-LN centroid displacement per phase
├── linear_probe_results.json ← accuracy for phase + ENSO probes
└── analysis_report.txt       ← summary with interpretations
```

**Interpreting your results:**
- High BSISO phase accuracy (> 50%) + significant ENSO displacement (z > 2) = model succeeded
- If ENSO displacement is not significant, consider: more epochs, lower temperature, Approach B preprocessing

---
*DDCS Project | jh9141@nyu.edu*